# 2. Baseline modeling — 3D U-Net + transformer

Thin driver around the vendored baseline in `scripts/` (see
`docs/0_coding_standards.md` for why that logic lives in `scripts/` rather
than `src/` for now). Two independent things this notebook can do,
controlled by `RUN_MODE`:

- **`"submission"`**: predict on the real competition `test/` set and write
  `submission.csv`, for upload. Defaults to the baseline author's public
  pretrained checkpoint (`thibautgoldsborough/cellmot-baseline-artifacts`)
  so a first submission doesn't require training anything ourselves —
  see `docs/1_instructions.md`.
- **`"train"`**: train our own checkpoint from scratch (a documented next
  experiment, not required for a first submission).

Not yet run: needs the competition data, which isn't downloaded locally —
run on Kaggle via `scripts/push_kaggle_kernel.sh baseline` (competition
mount + the public `cellmot-baseline-artifacts` dataset, which bundles a
working `repo/` alongside pretrained `weights/`, auto-detect — see the
Setup cell) or point `$CELLMOT_DATA_DIR` at a local copy.

## 1. Setup & Config

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    # Pin numpy/scipy/torch to whatever Kaggle's base image already has:
    # letting pip pick a newer numpy for zarr>=3.0.10 breaks the base
    # image's precompiled scipy (ImportError deep in scipy.spatial/
    # numpy._core -- an ABI mismatch between new numpy and the untouched
    # old scipy build). Separately, tracksdata depends on torch, and an
    # unpinned reinstall swaps out the base image's GPU-driver-matched
    # torch build for an incompatible one (`CUDA error: no kernel image is
    # available for execution on the device`). Both observed directly on
    # this competition's Kaggle image. Must also run before importing
    # numpy/matplotlib/torch below, for the same reason.
    import importlib.metadata

    _pinned = {
        pkg: importlib.metadata.version(pkg) for pkg in ("numpy", "scipy", "torch")
    }
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            *(f"{pkg}=={ver}" for pkg, ver in _pinned.items()),
            "zarr>=3.0.10", "tqdm", "polars", "pyscipopt",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )

    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )

    # The dataset mounts read-only, but the vendored scripts write
    # predictions/weights relative to their own file location
    # (scripts/dataspec.py) -- copy the repo to a writable location first.
    REPO_ROOT = Path("/kaggle/working/repo")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(ARTIFACTS_MOUNT / "repo", REPO_ROOT)
else:
    REPO_ROOT = Path.cwd().parent
    ARTIFACTS_MOUNT = None  # pretrained weights are Kaggle-only; train locally instead

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
RUN_MODE = "submission"  # "train" | "submission"

# --- "submission" mode: which weights to predict with ---------------------
USE_PRETRAINED = True  # True -> the public baseline checkpoint; False -> our own weights/ below
PRETRAINED_METHOD = "unet_transformer"
PRETRAINED_SPLIT = "0"

# --- "train" mode, and our-own-weights naming for "submission" mode -------
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3

# --- test-time detection/linking knobs (only used in "submission" mode) ---
# GT is sparse so the detector is poorly calibrated; ~0.99 scored best in the
# baseline author's sweep. ILP (global, flow-consistent linking) scored
# ~0.73 -> ~0.79 over the faster greedy linker in their notes.
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0


def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter.

    Sets PYTHONPATH=<REPO_ROOT>/src so `import tracking_cellmot` resolves in
    the subprocess -- unlike a local `uv run`, nothing here `pip install -e`s
    the package, so it's only importable via sys.path/PYTHONPATH.
    """
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT, env=env)


def resolve_weights() -> tuple[Path, str]:
    """Return (checkpoint path, method name) per USE_PRETRAINED."""
    if USE_PRETRAINED:
        if ARTIFACTS_MOUNT is None:
            raise FileNotFoundError(
                "USE_PRETRAINED=True but the cellmot-baseline-artifacts dataset isn't "
                "mounted -- add it as a data source, or set USE_PRETRAINED=False to use "
                "our own weights/ (requires RUN_MODE='train' first)."
            )
        split_dir = ARTIFACTS_MOUNT / "weights" / PRETRAINED_METHOD / f"split_{PRETRAINED_SPLIT}"
        return split_dir / "edge_predictor_best.pth", PRETRAINED_METHOD
    split_dir = REPO_ROOT / "weights" / METHOD / f"split_{SPLIT}"
    return split_dir / "edge_predictor_best.pth", METHOD


print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}")
if IS_KAGGLE:
    print(f"ARTIFACTS_MOUNT={ARTIFACTS_MOUNT}")

## 2. Train (optional — skip if `USE_PRETRAINED`)

Only runs in `RUN_MODE == "train"`. Not needed for a first submission (see
`USE_PRETRAINED` above) — this is how to train our own checkpoint to try to
beat the public baseline later.

In [ ]:
if RUN_MODE == "train":
    run(
        "scripts/train_unet_transformer.py",
        "--split", SPLIT,
        "--epochs", str(EPOCHS),
    )
    print(f"Trained weights: {REPO_ROOT}/weights/{METHOD}/split_{SPLIT}/edge_predictor_best.pth")

*Insight: fill in after running — training loss curve, whether it converged
in `EPOCHS` epochs, any stability issues.*

## 3. Predict on the competition test set

Only runs in `RUN_MODE == "submission"`. `predict_unet_transformer.py`
requires a `dataset_splits.json` listing which videos to predict — the real
`test/` directory doesn't ship one (that's a train-only, fold-splitting
concept), so build a synthetic one-fold file listing every test video
first, matching the approach in the baseline author's own public inference
notebook (`thibautgoldsborough/unet-baseline-inference-submission`).

In [ ]:
if RUN_MODE == "submission":
    import json

    test_stems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))
    print(f"{len(test_stems)} test videos under {TEST_DIR}")

    test_splits_file = REPO_ROOT / "kaggle_test_splits.json"
    test_splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}]))

    weights_path, predict_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(TEST_DIR),
        "--splits", str(test_splits_file),
        "--split", "0",
        "--method", predict_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

## 4. Build `submission.csv`

Flattens the predicted `.geff` graphs (one per test video) into the
competition's CSV schema — verified against the real `sample_submission.csv`
downloaded via the Kaggle CLI: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`,
one `node` row per detection and one `edge` row per link.

Inlined rather than calling `scripts/geffs_to_csv.py`: the artifacts
dataset's bundled `repo/` only includes what the baseline author's own
inference notebook needs (train/predict/dataspec), not this project's
extra conversion scripts (`geffs_to_csv.py`, `csv_to_geffs.py`,
`evaluate.py`) — confirmed missing on a real run. This mirrors the exact
logic in `scripts/geffs_to_csv.py`, kept in sync by hand for now.

In [ ]:
if RUN_MODE == "submission":
    import os

    import polars as pl
    import tracksdata as td

    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / predict_method / "split_0"
    submission_csv = Path("/kaggle/working/submission.csv") if IS_KAGGLE else REPO_ROOT / "submission.csv"

    def _graph_to_rows(graph, name: str) -> pl.DataFrame:
        """Flatten one graph into node rows then edge rows (submission schema)."""
        nodes = graph.node_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("node").alias("row_type"),
            pl.col("node_id").cast(pl.Int64),
            pl.col("t").cast(pl.Int64),
            pl.col("z").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("y").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("x").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.lit(-1, dtype=pl.Int64).alias("source_id"),
            pl.lit(-1, dtype=pl.Int64).alias("target_id"),
        )
        edges = graph.edge_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("edge").alias("row_type"),
            pl.lit(-1, dtype=pl.Int64).alias("node_id"),
            pl.lit(-1, dtype=pl.Int64).alias("t"),
            pl.lit(-1, dtype=pl.Int64).alias("z"),
            pl.lit(-1, dtype=pl.Int64).alias("y"),
            pl.lit(-1, dtype=pl.Int64).alias("x"),
            pl.col("source_id").cast(pl.Int64),
            pl.col("target_id").cast(pl.Int64),
        )
        return pl.concat([nodes, edges])

    geffs = sorted(predictions_dir.glob("*.geff"))
    frames = []
    for g in geffs:
        graph = td.graph.IndexedRXGraph.from_geff(str(g))
        graph = graph[0] if isinstance(graph, tuple) else graph
        frames.append(_graph_to_rows(graph, g.stem))
        print(f"{g.stem}: {graph.num_nodes()} nodes, {graph.num_edges()} edges")

    columns = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    table = pl.concat(frames) if frames else pl.DataFrame(schema=dict.fromkeys(columns, pl.Int64))
    table = table.with_row_index("id")
    table.write_csv(submission_csv)
    print(f"Wrote {table.height} rows from {len(geffs)} geffs to {submission_csv}")

## 5. (Optional) Validate methodology on a train fold

The real `test/` set has no local ground truth to score against. To
sanity-check the weights/detection/linking config *before* spending a
submission attempt, predict on a held-out **train** fold instead (real GT
available) and score locally with the competition's own metric
(`docs/1_instructions.md` / `metrics.md`). Builds its own deterministic
90/10 train/val split the same way `train_unet_transformer.py` does when no
`dataset_splits.json` is present, so it doesn't depend on a prior run.

In [ ]:
VALIDATE_ON_TRAIN_FOLD = False  # set True to sanity-check before submitting

if VALIDATE_ON_TRAIN_FOLD:
    import json
    import random

    stems = sorted(
        p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
        if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
    )
    random.Random(0).shuffle(stems)
    n_val = max(1, len(stems) // 10)
    train_splits_file = REPO_ROOT / "kaggle_train_splits.json"
    train_splits_file.write_text(json.dumps(
        [{"split": 0, "train": stems[n_val:], "test": stems[:n_val]}]
    ))
    print(f"{len(stems) - n_val} train / {n_val} val videos under {DATASET_PATH}")

    weights_path, validate_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(DATASET_PATH),
        "--splits", str(train_splits_file),
        "--split", "0",
        "--method", validate_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--evaluate",
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

*Insight: fill in after running — edge Jaccard, division Jaccard, final
score printed by `--evaluate`, and how it compares to the previous best
(record in `README.md`'s "Current best result" table).*

## Submitting to Kaggle

This competition scores a **file upload**, not a notebook rerun (see
`docs/1_instructions.md`) — after `RUN_MODE == "submission"` finishes on
Kaggle, download `submission.csv` from the kernel's Output tab, then
either upload it on the competition's Submit page, or from a shell with
Kaggle CLI access:

```bash
uv run kaggle competitions submit \
    -c biohub-cell-tracking-during-development \
    -f submission.csv \
    -m "unet_transformer split_0 pretrained, ILP, det-threshold 0.99"
```

Not run automatically from this notebook — submissions count against a
daily quota and should be a deliberate action, not a side effect of
re-running a cell.

## Findings / limitations / next experiment

- **Findings** (real run, 2026-07-21): completed end-to-end on Kaggle GPU
  (`device=cuda`, `NvidiaTeslaT4`) using the pretrained `unet_transformer`
  checkpoint. All 4 test videos predicted; `submission.csv` written with
  304,792 rows (164,682 node / 140,110 edge), schema verified against the
  real `sample_submission.csv`. Not yet uploaded to the leaderboard — that's
  a separate, deliberate action (see "Submitting to Kaggle" above).
- **Limitations**: the pretrained checkpoint (`unet_transformer`, split 0)
  wasn't trained to convergence per the baseline author's own notes — it's
  a starting point to beat, not a ceiling. `DET_THRESHOLD`/ILP weights
  above are their reported best settings, not necessarily ours.
- **Next** — see `docs/3_strategy.md` for the full prioritized roadmap,
  synthesized from six public reference notebooks (learned and
  classical, LB 0.73–0.897). Headline finding: every top score comes from
  the *same* detect → link → repair shape we already have — the gap isn't
  architecture, it's a deterministic **graph-repair** stage after ILP
  (motion relink, gap + gap2 closing, short-track pruning, trajectory
  smoothing) that this notebook doesn't implement yet. In order:
  1. Read `estimated_number_of_nodes` (now in `01_eda.ipynb`) to calibrate
     `DET_THRESHOLD` against an over-prediction budget.
  2. Add a graph-repair stage after the ILP predict step, each technique
     behind its own config flag (matching `USE_ILP`'s pattern), validated
     via `VALIDATE_ON_TRAIN_FOLD` before spending a submission.
  3. Division recovery is low-value in isolation (10% metric weight,
     `metrics.md`) — only after 1–2 are solid.
  4. Training our own checkpoint (`RUN_MODE = "train"`, `USE_PRETRAINED =
     False`) or D4 test-time augmentation are later-stage refinements.